# 🇸🇦 Arabic Piper TTS Fine-Tuning — Local Workstation (RTX 5090)

Tailored for local Linux GPU workstations (e.g. NVIDIA RTX 5090). Run cells sequentially.

### Sections:
1. ⚙️ Environment Setup & System Check
2. 📦 Dataset Download & Preparation
3. 🔊 Baseline Benchmark
4. 🏋️ Fine-Tuning Execution
5. 📊 Export & Evaluation

---
## ⚙️ Section 1: Environment Setup & System Check

In [ ]:
# 1.1 GPU Check
!nvidia-smi
import torch
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device Name : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

In [ ]:
# 1.2 Verify Directory Layout
from pathlib import Path
for d in ['datasets', 'processed', 'checkpoints', 'tensorboard', 'outputs', 'metrics']:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f'Verified local directory: ./{d}')

In [ ]:
# 1.3 Install System & Python Dependencies
import sys; print(f'Python Version: {sys.version}')
!pip install -q -r requirements.txt onnxscript onnx
print('\n✅ Local Python dependencies verified.')

In [ ]:
# 1.4 Install Piper Training Engine & Compile monotonic_align
import os, subprocess, sys, sysconfig
from pathlib import Path

PIPER_SRC = os.path.abspath('./piper_src') if os.path.exists('./piper_src') else '/tmp/piper'
if not os.path.exists(PIPER_SRC):
    os.system(f'git clone https://github.com/rhasspy/piper.git {PIPER_SRC}')

os.system('pip install -q cython setuptools "pytorch-lightning~=1.9.5" onnxscript onnx')
os.system(f'pip install -q --no-deps -e {PIPER_SRC}/src/python')

# Patch 1: PyTorch 2.6 weights_only checkpoint unpickling
main_py = Path(f'{PIPER_SRC}/src/python/piper_train/__main__.py')
if main_py.exists():
    content = main_py.read_text()
    patch = 'import pathlib, torch\ntry:\n    torch.serialization.add_safe_globals([pathlib.PosixPath, pathlib.WindowsPath])\nexcept Exception:\n    pass\n\n'
    if 'add_safe_globals' not in content:
        main_py.write_text(patch + content)
        print('✅ Patched piper_train/__main__.py for PyTorch 2.6 checkpoint loading.')

# Patch 2: Single-speaker dataset collate assertion in dataset.py
dataset_py = Path(f'{PIPER_SRC}/src/python/piper_train/vits/dataset.py')
if dataset_py.exists():
    ds_content = dataset_py.read_text()
    old_code = 'if utt.speaker_id is not None:'
    new_code = 'if self.is_multispeaker and (utt.speaker_id is not None):'
    if old_code in ds_content:
        ds_content = ds_content.replace(old_code, new_code)
        dataset_py.write_text(ds_content)
        print('✅ Patched piper_train/vits/dataset.py for single-speaker dataset collate.')

# Patch 3: Dynamic guard assertion in transforms.py for ONNX export
transforms_py = Path(f'{PIPER_SRC}/src/python/piper_train/vits/transforms.py')
if transforms_py.exists():
    tf_content = transforms_py.read_text()
    old_assert = 'assert (discriminant >= 0).all(), discriminant'
    new_assert = 'discriminant = torch.clamp(discriminant, min=0)'
    if old_assert in tf_content:
        tf_content = tf_content.replace(old_assert, new_assert)
        transforms_py.write_text(tf_content)
        print('✅ Patched piper_train/vits/transforms.py for ONNX tracing.')

# Patch 4: Force legacy TorchScript tracing (dynamo=False) in export_onnx.py
export_py = Path(f'{PIPER_SRC}/src/python/piper_train/export_onnx.py')
if export_py.exists():
    exp_content = export_py.read_text()
    if 'dynamo=' not in exp_content and 'torch.onnx.export(' in exp_content:
        exp_content = exp_content.replace('torch.onnx.export(', 'torch.onnx.export(dynamo=False, ')
        export_py.write_text(exp_content)
        print('✅ Patched piper_train/export_onnx.py to force dynamo=False.')

# Compile monotonic_align
mono_dir = Path(f'{PIPER_SRC}/src/python/piper_train/vits/monotonic_align')
out_subdir = mono_dir / 'monotonic_align'
out_subdir.mkdir(parents=True, exist_ok=True)

r1 = subprocess.run([sys.executable, '-m', 'cython', '-3', 'core.pyx'],
                    cwd=mono_dir, capture_output=True, text=True)
print(r1.stderr if r1.returncode != 0 else '✅ Cython: core.pyx → core.c')

py_inc = sysconfig.get_path('include')
suffix = sysconfig.get_config_var('EXT_SUFFIX')
out_so = str(out_subdir / f'core{suffix}')
r2 = subprocess.run(
    ['gcc', '-shared', '-fPIC', '-O2', f'-I{py_inc}', 'core.c', '-o', out_so],
    cwd=mono_dir, capture_output=True, text=True
)
print(r2.stderr if r2.returncode != 0 else f'✅ GCC: core.c → {out_so}')

print('\n✅ Local piper_train installed & patched!')

---
## 📦 Section 2: Dataset Download & Preparation

In [ ]:
# 2.1 Download Dataset & Base Checkpoint
!python scripts/download_dataset.py --config configs/experiment001.yaml --data-root .

In [ ]:
# 2.2 Prepare Dataset (wavs/ + metadata.csv + config.json + dataset.jsonl)
!python scripts/prepare_dataset.py --config configs/experiment001.yaml --data-root .

In [ ]:
# 2.3 Verify Processed Files
from pathlib import Path
processed_dir = Path('processed/experiment001')
required = ['config.json', 'dataset.jsonl', 'metadata.csv', 'wavs']
all_ok = True
for name in required:
    exists = (processed_dir / name).exists()
    print(f'  {"✅" if exists else "❌ MISSING"}  {name}')
    if not exists: all_ok = False
if all_ok:
    n_wav = len(list((processed_dir / 'wavs').glob('*.wav')))
    n_jl  = sum(1 for _ in open(processed_dir / 'dataset.jsonl'))
    print(f'\n✅ Dataset Ready — {n_wav} WAVs, {n_jl} phonemized entries.')

---
## 🔊 Section 3: Baseline Benchmark

In [ ]:
# 3.1 Download Base ONNX Model for Baseline Comparison
!mkdir -p checkpoints/base/
!wget -q -O checkpoints/base/ar_JO-kareem-medium.onnx \
    https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ar/ar_JO/kareem/medium/ar_JO-kareem-medium.onnx
!wget -q -O checkpoints/base/ar_JO-kareem-medium.onnx.json \
    https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ar/ar_JO/kareem/medium/ar_JO-kareem-medium.onnx.json
print('✅ Baseline ONNX model downloaded.')

In [ ]:
# 3.2 Benchmark Baseline Model
!python scripts/benchmark.py \
    --model checkpoints/base/ar_JO-kareem-medium.onnx \
    --model-config checkpoints/base/ar_JO-kareem-medium.onnx.json \
    --sentences benchmark/benchmark_sentences.txt \
    --output-dir outputs/baseline_benchmark

---
## 🏋️ Section 4: Fine-Tuning Execution

In [ ]:
# 4.1 Checkpoint Detection & Target Epoch Calculation
import os
from pathlib import Path

ckpt_dir = Path('checkpoints/experiment001')
ckpt_dir.mkdir(parents=True, exist_ok=True)
existing = sorted(ckpt_dir.rglob('*.ckpt'))

BASE_EPOCH = 5079
FINE_TUNE_EPOCHS = 50  # Additional fine-tuning epochs
TARGET_MAX_EPOCHS = BASE_EPOCH + FINE_TUNE_EPOCHS

if existing:
    resume_ckpt = str(existing[-1])
    print(f'✅ Resuming from fine-tuning checkpoint: {resume_ckpt}')
else:
    base_candidates = list(Path('checkpoints/base').rglob('*.ckpt'))
    resume_ckpt = str(base_candidates[0]) if base_candidates else ''
    print(f'🆕 Fine-tuning from base checkpoint (epoch {BASE_EPOCH} → {TARGET_MAX_EPOCHS}): {resume_ckpt}')

os.environ['RESUME_CKPT'] = resume_ckpt
os.environ['TARGET_MAX_EPOCHS'] = str(TARGET_MAX_EPOCHS)

In [ ]:
# 4.2 Run Fine-Tuning (Optimized for RTX 5090: Batch Size 32)
!python -m piper_train \
    --dataset-dir processed/experiment001 \
    --accelerator gpu \
    --devices 1 \
    --batch-size 32 \
    --validation-split 0.05 \
    --max_epochs $TARGET_MAX_EPOCHS \
    --checkpoint-epochs 5 \
    --default_root_dir checkpoints/experiment001 \
    --resume_from_checkpoint "$RESUME_CKPT"

---
## 📊 Section 5: Export & Evaluation

In [ ]:
# 5.1 Export Checkpoint to ONNX
from pathlib import Path
ckpt_dir = Path('checkpoints/experiment001')
ckpts = sorted(ckpt_dir.rglob('*.ckpt'))
config_json = Path('processed/experiment001/config.json')

if ckpts:
    ckpt = str(ckpts[-1])
    onnx = 'outputs/experiment001/ar_JO_finetuned.onnx'
    print(f'✅ Found checkpoint: {ckpt}')
    print(f'Exporting to ONNX: {onnx}')
    !python scripts/export_model.py \
        --checkpoint "{ckpt}" \
        --output-onnx "{onnx}" \
        --config-json "{config_json}"
else:
    print('❌ No checkpoint found in', ckpt_dir)

In [ ]:
# 5.2 Benchmark Fine-Tuned ONNX Model
from pathlib import Path
model_path = Path('outputs/experiment001/ar_JO_finetuned.onnx')
config_path = Path('outputs/experiment001/ar_JO_finetuned.onnx.json')
sentences_path = Path('benchmark/benchmark_sentences.txt')
out_dir = Path('outputs/finetuned_benchmark')

if model_path.exists():
    !python scripts/benchmark.py \
        --model "{model_path}" \
        --model-config "{config_path}" \
        --sentences "{sentences_path}" \
        --output-dir "{out_dir}"
else:
    print(f'❌ Fine-tuned model not found at {model_path}. Run Cell 5.1 first.')

In [ ]:
# 5.3 Side-by-Side Audio & RTF Comparison
import json
from pathlib import Path
from IPython.display import Audio, display, HTML

base_dir = Path('outputs/baseline_benchmark')
ft_dir   = Path('outputs/finetuned_benchmark')

base_report = base_dir / 'benchmark_report.json'
ft_report   = ft_dir / 'benchmark_report.json'

if base_report.exists() and ft_report.exists():
    b_data = json.loads(base_report.read_text())
    f_data = json.loads(ft_report.read_text())
    
    print('='*60)
    print(f"📊 Workstation Benchmark Summary:")
    print(f"   Baseline Average RTF   : {b_data.get('avg_rtf')}")
    print(f"   Fine-Tuned Average RTF: {f_data.get('avg_rtf')}")
    print('='*60)
    
    b_details = {item['id']: item for item in b_data.get('details', [])}
    f_details = {item['id']: item for item in f_data.get('details', [])}
    
    for item_id in sorted(f_details.keys()):
        text = f_details[item_id]['text']
        b_wav = base_dir / f"benchmark_{item_id:02d}.wav"
        f_wav = ft_dir / f"benchmark_{item_id:02d}.wav"
        
        display(HTML(f'<h4>Sample {item_id}: <i>"{text}"</i></h4>'))
        if b_wav.exists():
            display(HTML('<b>🔊 Baseline:</b>'))
            display(Audio(str(b_wav)))
        if f_wav.exists():
            display(HTML('<b>🎙️ Fine-Tuned:</b>'))
            display(Audio(str(f_wav)))
else:
    print('❌ Missing benchmark reports. Ensure Cells 3.2 and 5.2 completed.')